# Chapter 2: Data and Sampling Distributions

**Author:** Ruben Alfa Richie  
**Source:** Practical Statistics for Data Scientists (O'Reilly)

---

## Overview

Big data doesn't eliminate sampling - it makes it more important. This chapter covers:
- Random sampling and bias
- Sample statistics vs population parameters
- Sampling distributions
- Bootstrap resampling
- Standard distributions (Normal, t, Chi-square, F, Binomial, Poisson)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## 2.1 Random Sampling and Sample Bias

### Theory

**Population vs Sample:**
- **Population**: Entire group we want to study
- **Sample**: Subset we actually observe

**Random Sampling:**
- Each member has equal probability of selection
- Prevents systematic bias
- Allows statistical inference

**Types of Bias:**
1. **Selection Bias**: Non-random sampling
2. **Self-Selection Bias**: Participants volunteer (e.g., online reviews)
3. **Survivorship Bias**: Only "survivors" visible (missing failures)
4. **Non-Response Bias**: Certain groups don't respond

**Sample Size:**
- Larger = more precision
- But quality > quantity
- Biased large sample < unbiased small sample

In [ ]:
# Population vs Sample demonstration
population_size = 100000
population = np.random.normal(100, 15, population_size)

# Random samples of different sizes
sample_sizes = [10, 50, 100, 500, 1000]
sample_means = []

for size in sample_sizes:
    sample = np.random.choice(population, size, replace=False)
    sample_means.append(np.mean(sample))

pop_mean = np.mean(population)

print(f"Population Mean: {pop_mean:.2f}")
print("\nSample Means by Size:")
for size, mean in zip(sample_sizes, sample_means):
    error = abs(mean - pop_mean)
    print(f"n={size:4d}: {mean:.2f} (error: {error:.2f})")

# Visualize
plt.figure(figsize=(12, 5))
plt.plot(sample_sizes, sample_means, 'o-', markersize=10, linewidth=2)
plt.axhline(pop_mean, color='red', linestyle='--', linewidth=2, label='True Population Mean')
plt.xlabel('Sample Size')
plt.ylabel('Sample Mean')
plt.title('Sample Mean Approaches Population Mean as Sample Size Increases')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Biased vs Unbiased sampling
# Scenario: Population has two groups (A: 70%, B: 30%)
pop_A = np.random.normal(100, 10, 7000)
pop_B = np.random.normal(120, 10, 3000)
full_population = np.concatenate([pop_A, pop_B])

# Random (unbiased) sample
random_sample = np.random.choice(full_population, 200, replace=False)

# Biased sample (over-represents group B)
biased_sample = np.concatenate([
    np.random.choice(pop_A, 50, replace=False),
    np.random.choice(pop_B, 150, replace=False)
])

print(f"True Population Mean: {full_population.mean():.2f}")
print(f"Random Sample Mean: {random_sample.mean():.2f}")
print(f"Biased Sample Mean: {biased_sample.mean():.2f} ← BIASED!")

plt.figure(figsize=(12, 5))
plt.hist(full_population, bins=50, alpha=0.3, label='Population', density=True)
plt.hist(random_sample, bins=30, alpha=0.5, label='Random Sample', density=True)
plt.hist(biased_sample, bins=30, alpha=0.5, label='Biased Sample', density=True)
plt.xlabel('Value')
plt.ylabel('Density')
plt.title('Biased Sampling Distorts Distribution')
plt.legend()
plt.show()

## 2.2 Central Limit Theorem (CLT)

### Theory

**Central Limit Theorem** (most important theorem in statistics):

> The sampling distribution of the mean approaches a **normal distribution** as sample size increases, **regardless of the population's distribution**.

**Key Points:**
- Works for any population distribution
- Larger n → more normal
- Typically n ≥ 30 is sufficient
- Enables confidence intervals and hypothesis tests

**Standard Error (SE):**
$$SE = \frac{\sigma}{\sqrt{n}}$$
- Standard deviation of sampling distribution
- Decreases with √n
- Measures precision of estimate

In [ ]:
# Central Limit Theorem demonstration
# Start with NON-NORMAL population (exponential)
population = np.random.exponential(scale=2, size=100000)

print(f"Population Distribution: Exponential (highly right-skewed)")
print(f"Population Mean: {population.mean():.2f}")
print(f"Population Std: {population.std():.2f}")

# Draw many samples and calculate their means
sample_sizes = [5, 10, 30, 100]
n_samples = 10000

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Show population
axes[0].hist(population, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Population\n(Exponential - Skewed)')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

# Show sampling distributions for different n
for idx, n in enumerate(sample_sizes):
    sample_means = []
    for _ in range(n_samples):
        sample = np.random.choice(population, n, replace=True)
        sample_means.append(np.mean(sample))
    
    sample_means = np.array(sample_means)
    
    # Theoretical SE
    theoretical_se = population.std() / np.sqrt(n)
    actual_se = sample_means.std()
    
    axes[idx+1].hist(sample_means, bins=50, edgecolor='black', alpha=0.7, density=True)
    
    # Overlay normal curve
    x = np.linspace(sample_means.min(), sample_means.max(), 100)
    axes[idx+1].plot(x, stats.norm.pdf(x, population.mean(), theoretical_se), 
                     'r-', linewidth=2, label='Normal fit')
    
    axes[idx+1].set_title(f'Sample Means (n={n})\nSE={actual_se:.3f}')
    axes[idx+1].set_xlabel('Sample Mean')
    axes[idx+1].legend()

axes[5].axis('off')
plt.suptitle('Central Limit Theorem: Sampling Distribution Becomes Normal', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

print("\n✓ Even from skewed population, sample means become normally distributed!")

## 2.3 Bootstrap Resampling

### Theory

**Bootstrap** = powerful resampling technique

**Procedure:**
1. Draw sample of size n **with replacement** from observed data
2. Calculate statistic of interest (mean, median, etc.)
3. Repeat R times (e.g., R=1000)
4. Analyze distribution of bootstrap statistics

**Uses:**
- Estimate standard errors
- Construct confidence intervals
- Assess variability of any statistic
- No distributional assumptions needed

**Key Insight:** Sample is our "best guess" of population, so resampling from it simulates drawing from population.

In [ ]:
# Bootstrap example
observed_sample = np.random.exponential(scale=20, size=50)

print(f"Observed Sample Mean: {observed_sample.mean():.2f}")
print(f"Observed Sample Median: {np.median(observed_sample):.2f}")

# Bootstrap resampling
n_bootstrap = 10000
bootstrap_means = []
bootstrap_medians = []

for _ in range(n_bootstrap):
    # Resample WITH replacement
    bootstrap_sample = np.random.choice(observed_sample, size=len(observed_sample), replace=True)
    bootstrap_means.append(np.mean(bootstrap_sample))
    bootstrap_medians.append(np.median(bootstrap_sample))

bootstrap_means = np.array(bootstrap_means)
bootstrap_medians = np.array(bootstrap_medians)

# Calculate bootstrap standard errors
se_mean = bootstrap_means.std()
se_median = bootstrap_medians.std()

print(f"\nBootstrap SE of Mean: {se_mean:.2f}")
print(f"Bootstrap SE of Median: {se_median:.2f}")

# 95% confidence intervals
ci_mean = np.percentile(bootstrap_means, [2.5, 97.5])
ci_median = np.percentile(bootstrap_medians, [2.5, 97.5])

print(f"\n95% CI for Mean: [{ci_mean[0]:.2f}, {ci_mean[1]:.2f}]")
print(f"95% CI for Median: [{ci_median[0]:.2f}, {ci_median[1]:.2f}]")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(bootstrap_means, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(observed_sample.mean(), color='red', linewidth=2, label='Observed')
axes[0].axvline(ci_mean[0], color='green', linestyle='--', label='95% CI')
axes[0].axvline(ci_mean[1], color='green', linestyle='--')
axes[0].set_title('Bootstrap Distribution of Mean')
axes[0].set_xlabel('Mean')
axes[0].legend()

axes[1].hist(bootstrap_medians, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].axvline(np.median(observed_sample), color='red', linewidth=2, label='Observed')
axes[1].axvline(ci_median[0], color='green', linestyle='--', label='95% CI')
axes[1].axvline(ci_median[1], color='green', linestyle='--')
axes[1].set_title('Bootstrap Distribution of Median')
axes[1].set_xlabel('Median')
axes[1].legend()

plt.tight_layout()
plt.show()